# SerendibAI Gemma 4B – Colab inference dashboard

This notebook runs the production Q4_0 GGUF on a Colab GPU and provides a temporary public Gradio link for manual chat testing.

Before running, select **Runtime → Change runtime type → GPU**. A T4 or better is sufficient. The model download requires a Hugging Face token that has been granted access to `google/gemma-4-E4B-it-qat-q4_0-gguf`; the token is entered interactively and is never saved in this notebook.

In [ ]:
!nvidia-smi
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q --no-cache-dir --force-reinstall --no-binary llama-cpp-python llama-cpp-python
!pip install -q --no-cache-dir gradio huggingface_hub

In [ ]:
from getpass import getpass
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

MODEL_REPO = "google/gemma-4-E4B-it-qat-q4_0-gguf"
MODEL_FILE = "gemma-4-E4B_q4_0-it.gguf"

hf_token = getpass("Hugging Face token (input stays hidden): " )
model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    token=hf_token,
)

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,
    n_ctx=4096,
    n_batch=512,
    verbose=False,
)
print(f"Loaded {MODEL_REPO} on GPU.")

In [ ]:
import gradio as gr

DEFAULT_SYSTEM_PROMPT = "You are a helpful, concise assistant."

def respond(message, history, system_prompt, temperature, max_tokens):
    messages = [{"role": "system", "content": system_prompt}]
    for item in history:
        if item["role"] in {"user", "assistant"}:
            messages.append({"role": item["role"], "content": item["content"]})
    messages.append({"role": "user", "content": message})

    response = llm.create_chat_completion(
        messages=messages,
        temperature=float(temperature),
        max_tokens=int(max_tokens),
        stream=True,
    )
    answer = ""
    for chunk in response:
        answer += chunk["choices"][0]["delta"].get("content", "")
        yield answer

with gr.Blocks(title="SerendibAI Gemma 4B") as demo:
    gr.Markdown("# SerendibAI Gemma 4B\nCUDA-backed test dashboard. The shared link is temporary and public—do not enter secrets or customer data.")
    with gr.Accordion("Generation settings", open=False):
        system_prompt = gr.Textbox(value=DEFAULT_SYSTEM_PROMPT, label="System prompt", lines=3)
        with gr.Row():
            temperature = gr.Slider(0, 1.5, value=0.2, step=0.05, label="Temperature")
            max_tokens = gr.Slider(16, 512, value=160, step=16, label="Max new tokens")
    gr.ChatInterface(
        fn=respond,
        additional_inputs=[system_prompt, temperature, max_tokens],
        type="messages",
        examples=["Hello! Please introduce yourself in Sinhala.", "What can you help me with?"],
    )

demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)